# 下載資料集和套件

In [1]:
import os
import cv2
import pandas as pd
import random
import shutil
from glob import glob

In [2]:
!pip install ultralytics -qq # YOLO套件

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 28.1 MB/s eta 0:00:00


In [3]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
os.environ["KAGGLE_CONFIG_DIR"] = "/content/drive/MyDrive/.kaggle"

In [7]:
!kaggle competitions download -c taica-cvpdl-2025-hw-1 -p /content/drive/MyDrive/taica_hw1

 73% 62.0M/85.2M [00:00<00:00, 319MB/s]
100% 85.2M/85.2M [00:00<00:00, 329MB/s]


In [8]:
!cp /content/drive/MyDrive/taica_hw1/taica-cvpdl-2025-hw-1.zip .
!unzip -oq taica-cvpdl-2025-hw-1.zip -d /content/drive/MyDrive/taica_hw1

# 資料集檢查

In [9]:
img_dir = "/content/drive/MyDrive/taica_hw1/train/img"

files = os.listdir(img_dir)
print("共有", len(files), "張圖片")

img_path = os.path.join(img_dir, files[0])
img = cv2.imread(img_path)

print("img.shape =", img.shape)
h, w, c = img.shape
print(f"影像大小: {w}x{h}, 通道數: {c}")


共有 1266 張圖片
img.shape = (360, 640, 3)
影像大小: 640x360, 通道數: 3


## 切分資料集

In [10]:
base_dir = "/content/drive/MyDrive/taica_hw1"
train_img_dir = os.path.join(base_dir, "train/img")
gt_path = os.path.join(base_dir, "train/gt.txt")


label_dir = os.path.join(base_dir, "train/labels")
os.makedirs(label_dir, exist_ok=True)


sample_img = cv2.imread(os.path.join(train_img_dir, os.listdir(train_img_dir)[0]))
H, W, C = sample_img.shape
print(f"影像大小: {W}x{H}, 通道數: {C}")


df = pd.read_csv(gt_path, header=None)
df.columns = ["image_id", "x", "y", "w", "h"]


for _, row in df.iterrows():
    image_id = row["image_id"]
    x, y, w, h = row["x"], row["y"], row["w"], row["h"]

    x_center = (x + w / 2) / W
    y_center = (y + h / 2) / H
    w_norm = w / W
    h_norm = h / H

    filename = f"{int(image_id):08d}.txt"
    label_path = os.path.join(label_dir, filename)

    with open(label_path, "a") as f:
        f.write(f"0 {x_center} {y_center} {w_norm} {h_norm}\n")

print("標註已轉換完成:", label_dir)

train_out = os.path.join(base_dir, "yolo_train")
val_out   = os.path.join(base_dir, "yolo_val")

for d in [train_out, val_out]:
    os.makedirs(os.path.join(d, "images"), exist_ok=True)
    os.makedirs(os.path.join(d, "labels"), exist_ok=True)

images = glob(os.path.join(train_img_dir, "*.jpg"))
random.shuffle(images)

split_idx = int(0.8 * len(images))
train_imgs, val_imgs = images[:split_idx], images[split_idx:]

def move_files(img_list, out_dir):
    for img_path in img_list:
        fname = os.path.basename(img_path)
        label_path = os.path.join(label_dir, fname.replace(".jpg", ".txt"))
        shutil.copy(img_path, os.path.join(out_dir, "images", fname))
        if os.path.exists(label_path):
            shutil.copy(label_path, os.path.join(out_dir, "labels", fname.replace(".jpg", ".txt")))

move_files(train_imgs, train_out)
move_files(val_imgs, val_out)
print(f"切分完成 Train: {len(train_imgs)}, Val: {len(val_imgs)}")


影像大小: 640x360, 通道數: 3
標註已轉換完成: /content/drive/MyDrive/taica_hw1/train/labels
切分完成 Train: 1012, Val: 254


# 使用YOLO model


In [11]:
yaml_path = "/content/drive/MyDrive/taica_hw1/data.yaml"

with open(yaml_path, "w") as f:
    f.write(
        "train: /content/drive/MyDrive/taica_hw1/yolo_train/images\n"
        "val: /content/drive/MyDrive/taica_hw1/yolo_val/images\n"
        "test: /content/drive/MyDrive/taica_hw1/test/img\n\n"
        "nc: 1\n"
        "names: ['pig']\n"
    )

print('✅ data.yaml 已更新完成！')


✅ data.yaml 已更新完成！


In [12]:
!rm -f /content/drive/MyDrive/taica_hw1/yolo_train/images.cache
!rm -f /content/drive/MyDrive/taica_hw1/yolo_val/images.cache

## 訓練model

In [ ]:
model = YOLO("yolov10l.yaml")

model.train(
    data="/content/drive/MyDrive/taica_hw1/data.yaml",
    epochs=1,
    imgsz=640,
    batch=16,
    optimizer="adamw",
    cos_lr=True,
    patience=20,
    amp=True,
    weight_decay=0.05,
    pretrained=False,
    project="/content/drive/MyDrive/taica_hw1/runs",
)



## 預測 test set

In [ ]:
model = YOLO("/content/drive/MyDrive/taica_hw1/runs/train/weights/best.pt")

model.predict(
    source="/content/drive/MyDrive/taica_hw1/test/img",
    imgsz=640,
    conf=0.1,
    save=True,
    save_txt=True,
    save_conf=True,
    project="/content/drive/MyDrive/taica_hw1/preds",
    name="test"
)


# 將 labels 轉成 kaggle 競賽格式

In [ ]:
import os
import glob
import pandas as pd
from PIL import Image

IMG_W, IMG_H = 640, 360

pred_dir = "/content/drive/MyDrive/taica_hw1/preds/test/labels"
out_csv = "/content/drive/MyDrive/taica_hw1/submission.csv"

rows = []

for txt_file in sorted(glob.glob(os.path.join(pred_dir, "*.txt"))):
    img_id = os.path.splitext(os.path.basename(txt_file))[0]
    img_id = int(img_id)  # 轉成整數 ID (比賽要求)

    preds = []
    with open(txt_file, "r") as f:
        for line in f:
            cls, xc, yc, w, h, conf = map(float, line.strip().split())
            # YOLO 格式是 normalized，轉回像素
            x_left = (xc - w/2) * IMG_W
            y_top  = (yc - h/2) * IMG_H
            bw     = w * IMG_W
            bh     = h * IMG_H
            # 格式: conf x_left y_top width height class
            preds.append(f"{conf:.4f} {x_left:.0f} {y_top:.0f} {bw:.0f} {bh:.0f} {int(cls)}")

    pred_str = " ".join(preds)
    rows.append({"Image_ID": img_id, "PredictionString": pred_str})

df = pd.DataFrame(rows, columns=["Image_ID", "PredictionString"])
df.to_csv(out_csv, index=False)

print(f" Submission 檔案已存成: {out_csv}")
print(df.head(10))
